# ESPN API: exploración de datos de soccer

Misma API no oficial que el notebook de NFL, pero el soccer tiene dos diferencias importantes:

1. **Cada liga tiene un código** (`eng.1`, `esp.1`, `usa.1`...) y las rutas usan `soccer/{código}`.
2. **No hay semanas:** el scoreboard se pide por día (`YYYYMMDD`) o por mes (`YYYYMM`). Los rangos de fechas devuelven error 400, así que una temporada se arma mes por mes.

**Contenido**
1. Setup y catálogo de ligas
2. Scoreboard (goles y tarjetas incluidos)
3. Tabla de posiciones
4. Equipos y plantilla
5. Detalle de un partido (stats de equipo, eventos clave, alineaciones)
6. Temporada completa histórica
7. EDA: goles, local, empates
8. Modelo Poisson simple para probabilidades 1X2
9. Ideas para modelar

In [ ]:
import json
import time
import hashlib
from pathlib import Path

import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)

SITE = "https://site.api.espn.com/apis/site/v2/sports"          # scoreboard, teams, roster, summary
SITE_V2 = "https://site.api.espn.com/apis/v2/sports"            # standings completas
WEB = "https://site.web.api.espn.com/apis/common/v3/sports"     # atletas: gamelog, stats
CORE = "https://sports.core.api.espn.com/v2/sports"             # odds, probabilidades, detalle fino

CACHE_DIR = Path("espn_cache")
CACHE_DIR.mkdir(exist_ok=True)

session = requests.Session()


def get_json(url, params=None, use_cache=True, max_age_hours=6, retries=3, pause=0.3):
    """GET con cache en disco y reintentos.

    La API no es oficial: no hay key ni limites publicados, pero conviene
    cachear y no disparar cientos de requests seguidos.
    """
    params = params or {}
    key = hashlib.md5((url + json.dumps(params, sort_keys=True)).encode()).hexdigest()
    path = CACHE_DIR / f"{key}.json"

    if use_cache and path.exists():
        age_h = (time.time() - path.stat().st_mtime) / 3600
        if age_h < max_age_hours:
            return json.loads(path.read_text())

    last_err = None
    for attempt in range(retries):
        try:
            r = session.get(url, params=params, timeout=20)
            if r.status_code == 404:
                print(f"404: {r.url}")
                return {}
            r.raise_for_status()
            data = r.json()
            path.write_text(json.dumps(data))
            time.sleep(pause)
            return data
        except Exception as e:  # red, 5xx, json invalido
            last_err = e
            time.sleep(1.5 * (attempt + 1))
    raise RuntimeError(f"Fallo {url} {params}: {last_err}")


def peek(obj, depth=2, indent=0, max_items=12):
    """Imprime el 'esqueleto' de un JSON: llaves, tipos y largos. Util para explorar."""
    pad = "  " * indent
    if isinstance(obj, dict):
        for i, (k, v) in enumerate(obj.items()):
            if i >= max_items:
                print(f"{pad}... (+{len(obj) - max_items} llaves)")
                break
            kind = type(v).__name__
            extra = f" [{len(v)}]" if isinstance(v, (list, dict)) else f" = {str(v)[:40]!r}"
            print(f"{pad}{k}: {kind}{extra}")
            if depth > 0 and isinstance(v, (dict, list)) and v:
                peek(v, depth - 1, indent + 1, max_items)
    elif isinstance(obj, list) and obj:
        print(f"{pad}[0] de {len(obj)}:")
        peek(obj[0], depth, indent + 1, max_items)


def dig(obj, *path, default=None):
    """Acceso seguro a llaves anidadas: dig(ev, 'competitions', 0, 'venue', 'fullName')."""
    cur = obj
    for p in path:
        try:
            cur = cur[p]
        except (KeyError, IndexError, TypeError):
            return default
    return cur


def to_num(x):
    """Convierte '24' o '3/10' o '--' a numero cuando se puede."""
    try:
        return float(x)
    except (TypeError, ValueError):
        return np.nan


def parse_standings(data):
    """Aplana standings: recorre children (conferencias, divisiones, grupos) de forma recursiva."""
    rows = []

    def walk(node, group=None):
        name = node.get("name") or node.get("abbreviation") or group
        entries = dig(node, "standings", "entries", default=[]) or []
        for e in entries:
            row = {
                "group": name,
                "team_id": dig(e, "team", "id"),
                "team": dig(e, "team", "displayName"),
                "abbr": dig(e, "team", "abbreviation"),
            }
            for s in e.get("stats", []):
                col = s.get("name") or s.get("type")
                if not col:
                    continue
                val = s.get("value")
                row[col] = val if val is not None else s.get("displayValue")
            rows.append(row)
        for child in node.get("children", []) or []:
            walk(child, name)

    walk(data)
    return pd.DataFrame(rows)

In [ ]:
LEAGUES = {
    "eng.1": "Premier League",
    "esp.1": "LaLiga",
    "ita.1": "Serie A",
    "ger.1": "Bundesliga",
    "fra.1": "Ligue 1",
    "usa.1": "MLS",
    "mex.1": "Liga MX",
    "uefa.champions": "Champions League",
}

LEAGUE = "eng.1"                       # cambia aqui la liga
HIST_SEASON_MONTHS = pd.period_range("2025-08", "2026-05", freq="M")   # temporada 2025-26
CURRENT_SEASON = 2026                  # ESPN usa el año de inicio: 2026 = temporada 2026-27 en Europa

BASE = f"{SITE}/soccer/{LEAGUE}"
print(LEAGUES[LEAGUE], "|", BASE)

## 1. Scoreboard

Sin parámetros devuelve la jornada actual. En soccer cada competidor trae `score`, `form` (racha reciente) y `records`, y la competición trae `details`: la lista de goles, tarjetas y penales con minuto y jugador.

In [ ]:
sb = get_json(f"{BASE}/scoreboard")
peek(sb, depth=1)
if sb.get("events"):
    print("\nEstructura de un evento:")
    peek(sb["events"][0], depth=2)

In [ ]:
def parse_scoreboard(data):
    rows = []
    for ev in data.get("events", []):
        comp = dig(ev, "competitions", 0, default={})
        teams = {c.get("homeAway"): c for c in comp.get("competitors", [])}
        home, away = teams.get("home", {}), teams.get("away", {})
        odds = dig(comp, "odds", 0, default={}) or {}
        rows.append({
            "event_id": ev.get("id"),
            "date": ev.get("date"),
            "name": ev.get("shortName") or ev.get("name"),
            "status": dig(comp, "status", "type", "description") or dig(ev, "status", "type", "description"),
            "completed": dig(comp, "status", "type", "completed", default=False),
            "home": dig(home, "team", "displayName"),
            "away": dig(away, "team", "displayName"),
            "home_id": dig(home, "team", "id"),
            "away_id": dig(away, "team", "id"),
            "home_goals": to_num(home.get("score")),
            "away_goals": to_num(away.get("score")),
            "home_form": home.get("form"),
            "away_form": away.get("form"),
            "venue": dig(comp, "venue", "fullName"),
            "attendance": comp.get("attendance"),
            "odds_details": odds.get("details"),
            "over_under": odds.get("overUnder"),
        })
    df = pd.DataFrame(rows)
    if not df.empty:
        df["date"] = pd.to_datetime(df["date"], utc=True)
    return df


def parse_details(data):
    # Goles, tarjetas y penales del scoreboard (competitions[0].details)
    rows = []
    for ev in data.get("events", []):
        comp = dig(ev, "competitions", 0, default={})
        names = {c["team"]["id"]: c["team"].get("displayName")
                 for c in comp.get("competitors", []) if c.get("team")}
        for d in comp.get("details", []) or []:
            rows.append({
                "event_id": ev.get("id"),
                "minute": dig(d, "clock", "displayValue"),
                "type": dig(d, "type", "text"),
                "team": names.get(dig(d, "team", "id")),
                "player": dig(d, "athletesInvolved", 0, "displayName"),
                "goal": d.get("scoringPlay"),
                "penalty": d.get("penaltyKick"),
                "own_goal": d.get("ownGoal"),
                "yellow": d.get("yellowCard"),
                "red": d.get("redCard"),
            })
    return pd.DataFrame(rows)


matches_now = parse_scoreboard(sb)
display(matches_now)
parse_details(sb).head(20)

## 2. Tabla de posiciones

Igual que en NFL, la tabla completa sale de `/apis/v2/sports/soccer/{liga}/standings`. En ligas normales hay un solo grupo; en Champions vienen varios `children` (uno por grupo o fase de liga).

In [ ]:
st_raw = get_json(f"{SITE_V2}/soccer/{LEAGUE}/standings", params={"season": CURRENT_SEASON})
table = parse_standings(st_raw)
print("Columnas:", list(table.columns))
cols = [c for c in ["group", "rank", "team", "gamesPlayed", "wins", "ties", "losses",
                    "pointsFor", "pointsAgainst", "pointDifferential", "points"]
        if c in table.columns]
table[cols].sort_values("rank" if "rank" in cols else cols[0])

## 3. Equipos y plantilla

`/teams` lista los clubes de la liga y `/teams/{id}/roster` la plantilla. En soccer el roster viene como lista plana de atletas.

In [ ]:
teams_raw = get_json(f"{BASE}/teams")
teams = pd.DataFrame([
    {"team_id": t["team"].get("id"),
     "abbr": t["team"].get("abbreviation"),
     "name": t["team"].get("displayName"),
     "logo": dig(t, "team", "logos", 0, "href")}
    for t in dig(teams_raw, "sports", 0, "leagues", 0, "teams", default=[])
])
display(teams)

TEAM_ID = teams["team_id"].iloc[0]
roster_raw = get_json(f"{BASE}/teams/{TEAM_ID}/roster")
athletes = roster_raw.get("athletes", [])
if athletes and "items" in athletes[0]:          # por si viene agrupado
    athletes = [a for g in athletes for a in g.get("items", [])]
roster = pd.DataFrame([{
    "athlete_id": a.get("id"),
    "name": a.get("displayName") or a.get("fullName"),
    "pos": dig(a, "position", "abbreviation"),
    "jersey": a.get("jersey"),
    "age": a.get("age"),
    "nationality": a.get("citizenship"),
} for a in athletes])
roster

## 4. Detalle de un partido: `summary`

Llaves útiles en soccer:

* `boxscore.teams`: posesión, tiros, tiros al arco, corners, faltas, pases
* `keyEvents`: goles, tarjetas, cambios y VAR con minuto
* `rosters`: alineaciones con titulares, cambios y stats individuales
* `headToHeadGames`, `standings`, `odds`, `gameInfo`

In [ ]:
done = matches_now[matches_now["completed"]] if not matches_now.empty else matches_now
if done.empty:
    first_month = HIST_SEASON_MONTHS[0].strftime("%Y%m")
    done = parse_scoreboard(get_json(f"{BASE}/scoreboard", params={"dates": first_month, "limit": 500}))
    done = done[done["completed"]]
EVENT_ID = done["event_id"].iloc[0]
print("Partido:", done["name"].iloc[0], "| event_id:", EVENT_ID)

summ = get_json(f"{BASE}/summary", params={"event": EVENT_ID})
peek(summ, depth=0)

In [ ]:
def team_stats(summary):
    rows = []
    for t in dig(summary, "boxscore", "teams", default=[]):
        row = {"team": dig(t, "team", "displayName")}
        for s in t.get("statistics", []):
            row[s.get("name") or s.get("label")] = s.get("displayValue")
        rows.append(row)
    return pd.DataFrame(rows).set_index("team").T if rows else pd.DataFrame()


key_events = pd.DataFrame([{
    "minute": dig(k, "clock", "displayValue"),
    "type": dig(k, "type", "text"),
    "team": dig(k, "team", "displayName"),
    "player": dig(k, "participants", 0, "athlete", "displayName"),
    "text": k.get("text") or k.get("shortText"),
} for k in summ.get("keyEvents", [])])

lineups = pd.DataFrame([{
    "team": dig(r, "team", "displayName"),
    "player": dig(p, "athlete", "displayName"),
    "athlete_id": dig(p, "athlete", "id"),
    "pos": dig(p, "position", "abbreviation"),
    "starter": p.get("starter"),
    "subbed_in": p.get("subbedIn"),
    "subbed_out": p.get("subbedOut"),
    **{s.get("name"): s.get("value") for s in p.get("stats", []) or []},
} for r in summ.get("rosters", []) for p in r.get("roster", [])])

display(team_stats(summ))
display(key_events)
lineups.head(25)

## 5. Temporada completa

Recorremos la temporada mes por mes con `dates=YYYYMM` y `limit=500`. Son unos 10 requests por liga, cacheados por 30 días.

In [ ]:
def load_season(months, league=LEAGUE):
    frames = []
    for m in months:
        data = get_json(f"{SITE}/soccer/{league}/scoreboard",
                        params={"dates": m.strftime("%Y%m"), "limit": 500},
                        max_age_hours=24 * 30)
        frames.append(parse_scoreboard(data))
    df = pd.concat([f for f in frames if not f.empty], ignore_index=True)
    return df.drop_duplicates("event_id").sort_values("date")


season = load_season(HIST_SEASON_MONTHS)
season = season[season["completed"]].copy()
season["result"] = np.select(
    [season["home_goals"] > season["away_goals"], season["home_goals"] < season["away_goals"]],
    ["H", "A"], default="D")
season["total_goals"] = season["home_goals"] + season["away_goals"]
season.to_csv(f"soccer_{LEAGUE}_{HIST_SEASON_MONTHS[0].year}.csv", index=False)
print(season.shape)
season.head()

## 6. EDA

Lo básico que cualquier modelo de soccer tiene que capturar: qué tan seguido gana el local, la tasa de empates (que hace al soccer más difícil de predecir que la NFL) y la distribución de goles, que se parece mucho a una Poisson.

In [ ]:
print(f"Partidos: {len(season)}")
print(season["result"].value_counts(normalize=True).rename({"H": "Local", "D": "Empate", "A": "Visita"}).round(3))
print(f"\nGoles por partido: {season['total_goals'].mean():.2f}")
print(f"Over 2.5: {(season['total_goals'] > 2.5).mean():.1%}")

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
season["total_goals"].value_counts().sort_index().plot.bar(ax=axes[0], color="#4C72B0")
axes[0].set_title("Goles totales por partido")

score_grid = pd.crosstab(season["home_goals"].clip(upper=5), season["away_goals"].clip(upper=5),
                         normalize=True)
im = axes[1].imshow(score_grid, cmap="Blues")
axes[1].set_xticks(range(len(score_grid.columns)), score_grid.columns.astype(int))
axes[1].set_yticks(range(len(score_grid.index)), score_grid.index.astype(int))
axes[1].set_xlabel("goles visita")
axes[1].set_ylabel("goles local")
axes[1].set_title("Frecuencia de marcadores (5 = 5+)")
plt.colorbar(im, ax=axes[1])
plt.tight_layout()
plt.show()

## 7. Modelo Poisson simple

Es el punto de partida clásico (Maher, 1982). La idea:

* Cada equipo tiene una **fuerza de ataque** (cuánto anota respecto al promedio de la liga) y una **fuerza de defensa** (cuánto recibe respecto al promedio).
* Los goles esperados del local = promedio de goles de local × ataque del local × defensa del visitante. Igual para la visita.
* Si los goles siguen una Poisson con esa media, la probabilidad de cada marcador es el producto de las dos Poisson, y sumando celdas sale la probabilidad de local, empate y visita.

Es simple a propósito: no pondera partidos recientes ni corrige la dependencia entre marcadores bajos (Dixon y Coles), pero sirve como baseline para comparar contra las cuotas.

In [ ]:
from math import exp, factorial

avg_home = season["home_goals"].mean()
avg_away = season["away_goals"].mean()

h = season.groupby("home").agg(hs=("home_goals", "mean"), hc=("away_goals", "mean"))
a = season.groupby("away").agg(as_=("away_goals", "mean"), ac=("home_goals", "mean"))
strength = h.join(a, how="outer")
strength["att_home"] = strength["hs"] / avg_home
strength["def_home"] = strength["hc"] / avg_away
strength["att_away"] = strength["as_"] / avg_away
strength["def_away"] = strength["ac"] / avg_home


def poisson_pmf(k, lam):
    return exp(-lam) * lam ** k / factorial(k)


def predict(home, away, max_goals=8):
    lam_h = avg_home * strength.loc[home, "att_home"] * strength.loc[away, "def_away"]
    lam_a = avg_away * strength.loc[away, "att_away"] * strength.loc[home, "def_home"]
    grid = np.array([[poisson_pmf(i, lam_h) * poisson_pmf(j, lam_a)
                      for j in range(max_goals)] for i in range(max_goals)])
    return {
        "xG_home": round(lam_h, 2), "xG_away": round(lam_a, 2),
        "p_home": grid[np.triu_indices(max_goals, 1)[::-1]].sum().round(3),
        "p_draw": np.trace(grid).round(3),
        "p_away": grid[np.triu_indices(max_goals, 1)].sum().round(3),
        "p_over_2_5": round(1 - sum(grid[i, j] for i in range(max_goals)
                                    for j in range(max_goals) if i + j <= 2), 3),
    }


strength[["att_home", "def_home", "att_away", "def_away"]].round(2).sort_values("att_home", ascending=False)

In [ ]:
# Probabilidades para los partidos de la jornada actual (solo equipos con historia)
known = set(strength.index)
preds = []
for _, m in matches_now.iterrows():
    if m["home"] in known and m["away"] in known:
        preds.append({"match": f"{m['home']} vs {m['away']}", **predict(m["home"], m["away"])})
pd.DataFrame(preds) if preds else print("Sin partidos con equipos conocidos en la jornada actual")

In [ ]:
# Backtest rapido (con fuga de informacion: las fuerzas se estimaron con los mismos partidos).
# Sirve para ver calibracion, no para medir rendimiento real.
bt = season.apply(lambda r: pd.Series(predict(r["home"], r["away"])), axis=1)
bt["result"] = season["result"].values
probs = bt[["p_home", "p_draw", "p_away"]].to_numpy()
actual = pd.get_dummies(bt["result"]).reindex(columns=["H", "D", "A"], fill_value=0).to_numpy()
brier = ((probs - actual) ** 2).sum(axis=1).mean()
print(f"Brier score 1X2: {brier:.3f}  (azar uniforme = 0.667)")

## 8. Ideas para modelar

* **Corregir la fuga:** estimar fuerzas solo con partidos anteriores a cada fecha (walk forward) y medir el Brier fuera de muestra.
* **Ponderar por recencia** y agregar la corrección de Dixon y Coles para marcadores 0-0, 1-0, 0-1 y 1-1.
* **Features de `summary`:** tiros al arco y posesión explican el rendimiento mejor que los goles, que tienen mucho ruido.
* **Comparar contra cuotas:** convertir `odds_details` a probabilidades implícitas y ver dónde difiere el modelo.